# 18 — Calibrate the dense long-video decoder

This CPU notebook keeps the frozen **V6-C** gate and exercise classifier. It
changes only the temporal decoder:

1. predicts every 8 frames instead of every 32 frames;
2. uses hysteresis so an action is easier to keep than to start;
3. removes implausibly short action runs;
4. joins short `unknown` gaps between two runs of the same exercise.

The parameter grid is scored equally across three creator videos excluded from
all V6 training, validation, and internal-test splits. The locked GainsByBrains
test is never loaded. Historical timeline notes still require a visual
spot-check, so the selected settings are **provisional**.

Use **Colab CPU**. A T4 is unnecessary. The final cell downloads
`calorie_v6_dense_decoder_report.zip`.


In [ ]:
%pip install -q torch numpy pandas==2.2.3 scikit-learn matplotlib seaborn


In [ ]:
from pathlib import Path
import json
import re
import shutil
import urllib.request
import zipfile
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, recall_score
from torch import nn

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SEED = 42
TARGET_FPS = 15
WINDOW = 64
DENSE_STRIDE = 8
RETIRED_CLASSES = {"plank"}
CALIBRATION_GROUPS = {
    "youtube:oLsonHQJOUc",
    "youtube:c4TomZjzobY",
    "youtube:W2Bm9dfQjbk",
}
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cpu")

WORK_ROOT = Path.cwd() / "calorie_dense_decoder_work"
DOWNLOAD_ROOT = WORK_ROOT / "downloads"
EXTRACT_ROOT = WORK_ROOT / "extracted"
REPORT_ROOT = WORK_ROOT / "calorie_v6_dense_decoder_report"
for path in [DOWNLOAD_ROOT, EXTRACT_ROOT, REPORT_ROOT]:
    path.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)
print("Dense prediction step:", round(DENSE_STRIDE / TARGET_FPS, 3), "seconds")


## Download compact frozen evidence

This downloads one trained V6-C report, the independent creator landmark
archive, and Notebook 17's calibration summary. No raw video or MediaPipe model
is needed.


In [ ]:
RAW = "https://raw.githubusercontent.com/bariscelikk1/cALorie/feat/tcn-rebuild/ml/artifacts"
ASSETS = {
    "v6c_report": f"{RAW}/v6c/model/calorie_v6c_kinematic_features_tcn_report.zip",
    "calibration_landmarks": f"{RAW}/v2/landmarks/calorie_youtube_mixed_landmarks_v3.zip",
    "calibration_report": f"{RAW}/calibration/calorie_v6_gate_calibration_report.zip",
}


def download(name, url):
    destination = DOWNLOAD_ROOT / Path(url).name
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=300) as response, destination.open("wb") as output:
        shutil.copyfileobj(response, output)
    print(name, "->", destination.name, round(destination.stat().st_size / 1e6, 2), "MB")
    return destination


downloaded = {name: download(name, url) for name, url in ASSETS.items()}
for name, archive_path in downloaded.items():
    destination = EXTRACT_ROOT / name
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir()
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(destination)

calibration_summary_path = next((EXTRACT_ROOT / "calibration_report").rglob("calibration_summary.json"))
calibration_summary = json.loads(calibration_summary_path.read_text())
GATE_ENTER = float(calibration_summary["thresholds"]["v6c"])
print("Frozen provisional V6-C gate threshold:", GATE_ENTER)


## Rebuild dense timelines

Each labelled segment is resampled to 15 FPS. A 64-frame window is centred
every 8 frames (about 0.53 seconds), including near segment boundaries through
edge padding. Plank remains retired and maps to `unknown`.


In [ ]:
def resample_sequence(sequence, source_fps, target_fps=TARGET_FPS):
    if not source_fps or abs(float(source_fps) - target_fps) < 0.1:
        return sequence
    duration = (len(sequence) - 1) / float(source_fps)
    target_length = max(2, int(round(duration * target_fps)) + 1)
    old_x = np.linspace(0, duration, len(sequence))
    new_x = np.linspace(0, duration, target_length)
    flat = sequence.reshape(len(sequence), -1)
    sampled = np.stack([
        np.interp(new_x, old_x, flat[:, column]) for column in range(flat.shape[1])
    ], axis=1)
    return sampled.reshape(target_length, 17, 4).astype(np.float32)


def centered_window(sequence, center):
    left = WINDOW // 2
    indices = np.arange(center - left, center - left + WINDOW)
    indices = np.clip(indices, 0, len(sequence) - 1)
    return sequence[indices]


def parse_segment_start(value):
    match = re.search(r"#([0-9.]+)-([0-9.]+)$", str(value))
    if not match:
        raise ValueError(f"Cannot parse timeline interval: {value}")
    return float(match.group(1)), float(match.group(2))

landmark_root = EXTRACT_ROOT / "calibration_landmarks"
manifest_path = next(landmark_root.rglob("manifest.csv"))
manifest = pd.read_csv(manifest_path).fillna("")
manifest["npz_path"] = manifest.output_path.map(lambda value: str(manifest_path.parent / value))
manifest = manifest[
    (manifest.status == "ok")
    & manifest.npz_path.map(lambda value: Path(value).exists())
    & manifest.group_id.isin(CALIBRATION_GROUPS)
].copy()
assert set(manifest.group_id) == CALIBRATION_GROUPS

windows, labels, groups, clip_ids, centers_s = [], [], [], [], []
for row in manifest.itertuples(index=False):
    with np.load(row.npz_path, allow_pickle=False) as item:
        sequence = item["landmarks"].astype(np.float32)
    sequence = resample_sequence(sequence, getattr(row, "source_fps", TARGET_FPS) or TARGET_FPS)
    segment_start, segment_end = parse_segment_start(row.original_path)
    centers = list(range(0, len(sequence), DENSE_STRIDE))
    if centers[-1] != len(sequence) - 1:
        centers.append(len(sequence) - 1)
    label = "unknown" if row.canonical_class in RETIRED_CLASSES else row.canonical_class
    for center in centers:
        windows.append(centered_window(sequence, center))
        labels.append(label)
        groups.append(row.group_id)
        clip_ids.append(row.clip_id)
        centers_s.append(min(segment_end, segment_start + center / TARGET_FPS))

X = np.asarray(windows, dtype=np.float32)
labels = np.asarray(labels)
groups = np.asarray(groups)
clip_ids = np.asarray(clip_ids)
centers_s = np.asarray(centers_s, dtype=np.float32)
order = np.lexsort((centers_s, groups))
X, labels, groups, clip_ids, centers_s = [item[order] for item in [X, labels, groups, clip_ids, centers_s]]

split_path = next((EXTRACT_ROOT / "v6c_report").rglob("split_assignments.csv"))
assert not (set(groups) & set(pd.read_csv(split_path).group_id)), "Calibration leakage into V6-C"
print("Segments:", len(manifest))
print("Dense windows:", len(X))
display(pd.crosstab(pd.Series(groups, name="creator"), pd.Series(labels, name="label")))


## Load frozen V6-C and predict

V6-C receives normalized x/y, velocity, directed bone vectors, and joint
angles. Training does not occur here.


In [ ]:
BONE_EDGES = [
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10), (5, 11),
    (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16),
]
ANGLE_TRIPLETS = [
    (5, 7, 9), (6, 8, 10), (7, 5, 11), (8, 6, 12),
    (5, 11, 13), (6, 12, 14), (11, 13, 15), (12, 14, 16),
]


def features_v6c(sample):
    xy = sample[:, :, :2].astype(np.float32)
    velocity = np.zeros_like(xy)
    velocity[1:] = xy[1:] - xy[:-1]
    bones = np.stack([xy[:, child] - xy[:, parent] for parent, child in BONE_EDGES], axis=1)
    angles = []
    for first, center, last in ANGLE_TRIPLETS:
        a = xy[:, first] - xy[:, center]
        b = xy[:, last] - xy[:, center]
        denominator = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1)
        cosine = np.sum(a * b, axis=1) / np.maximum(denominator, 1e-6)
        angles.append(np.arccos(np.clip(cosine, -1.0, 1.0)) / np.pi)
    return np.concatenate([
        xy.reshape(WINDOW, -1), velocity.reshape(WINDOW, -1),
        bones.reshape(WINDOW, -1), np.stack(angles, axis=1),
    ], axis=1).astype(np.float32)


class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation, dropout=0.2):
        super().__init__()
        padding = dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
        )
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return torch.relu(self.net(x) + self.residual(x))


class SmallTCN(nn.Module):
    def __init__(self, input_features, num_classes):
        super().__init__()
        channels = [64, 96, 128, 128]
        blocks, current = [], input_features
        for level, output in enumerate(channels):
            blocks.append(TemporalBlock(current, output, dilation=2 ** level))
            current = output
        self.features = nn.Sequential(*blocks)
        self.classifier = nn.Linear(current, num_classes)

    def forward(self, x):
        return self.classifier(self.features(x).mean(dim=-1))

checkpoint_path = next((EXTRACT_ROOT / "v6c_report").rglob("two_stage_tcn_v6c.pt"))
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
SUPPORTED_CLASSES = checkpoint["supported_classes"]
gate = SmallTCN(checkpoint["input_features"], 2).to(DEVICE)
exercise = SmallTCN(checkpoint["input_features"], len(SUPPORTED_CLASSES)).to(DEVICE)
gate.load_state_dict(checkpoint["gate_state_dict"])
exercise.load_state_dict(checkpoint["exercise_state_dict"])
gate.eval(); exercise.eval()

features = np.stack([features_v6c(window) for window in X]).transpose(0, 2, 1)
gate_parts, exercise_parts = [], []
with torch.no_grad():
    for start in range(0, len(features), 256):
        batch = torch.from_numpy(features[start:start + 256]).to(DEVICE)
        gate_parts.append(gate(batch).softmax(1).cpu().numpy())
        exercise_parts.append(exercise(batch).softmax(1).cpu().numpy())
gate_probability = np.concatenate(gate_parts)[:, 1]
exercise_probability = np.concatenate(exercise_parts)
exercise_ids = exercise_probability.argmax(1)
exercise_top = np.asarray([SUPPORTED_CLASSES[index] for index in exercise_ids])
exercise_confidence = exercise_probability.max(1)
raw_prediction = np.where(gate_probability >= GATE_ENTER, exercise_top, "unknown")
print("Inference complete:", len(raw_prediction), "dense windows")


## Calibrate hysteresis, duration, and gap merging

The gate entry threshold remains fixed from Notebook 17. The grid changes only
how predictions persist over time. Selection uses creator-equal Macro F1, then
the worst creator score, so the longest source cannot dominate.


In [ ]:
STEP_SECONDS = DENSE_STRIDE / TARGET_FPS
present_labels = sorted(set(labels))


def run_lengths(values):
    starts = [0]
    for index in range(1, len(values)):
        if values[index] != values[index - 1]:
            starts.append(index)
    starts.append(len(values))
    return [(starts[i], starts[i + 1], values[starts[i]]) for i in range(len(starts) - 1)]


def postprocess_runs(values, min_duration_s, merge_gap_s):
    result = np.asarray(values, dtype=object).copy()
    max_gap = int(round(merge_gap_s / STEP_SECONDS))
    min_run = max(1, int(round(min_duration_s / STEP_SECONDS)))
    if max_gap > 0:
        for start, end, label in run_lengths(result):
            if label == "unknown" and end - start <= max_gap and start > 0 and end < len(result):
                if result[start - 1] == result[end] and result[start - 1] != "unknown":
                    result[start:end] = result[start - 1]
    if min_duration_s > 0:
        for start, end, label in run_lengths(result):
            if label != "unknown" and end - start < min_run:
                result[start:end] = "unknown"
    if max_gap > 0:
        for start, end, label in run_lengths(result):
            if label == "unknown" and end - start <= max_gap and start > 0 and end < len(result):
                if result[start - 1] == result[end] and result[start - 1] != "unknown":
                    result[start:end] = result[start - 1]
    return result


def decode_creator(indices, exit_threshold, exercise_enter, exercise_hold, min_duration_s, merge_gap_s):
    state = "unknown"
    values = []
    for index in indices:
        top_label = exercise_top[index]
        top_confidence = float(exercise_confidence[index])
        supported = float(gate_probability[index])
        if state == "unknown":
            state = top_label if supported >= GATE_ENTER and top_confidence >= exercise_enter else "unknown"
        else:
            current_id = SUPPORTED_CLASSES.index(state)
            current_confidence = float(exercise_probability[index, current_id])
            if supported >= exit_threshold and current_confidence >= exercise_hold:
                pass
            elif supported >= GATE_ENTER and top_confidence >= exercise_enter:
                state = top_label
            else:
                state = "unknown"
        values.append(state)
    return postprocess_runs(values, min_duration_s, merge_gap_s)


def score_predictions(prediction):
    creator_scores = [
        f1_score(
            labels[groups == group], prediction[groups == group],
            labels=sorted(set(labels[groups == group])), average="macro", zero_division=0,
        ) for group in sorted(CALIBRATION_GROUPS)
    ]
    return {
        "accuracy": accuracy_score(labels, prediction),
        "macro_f1_present_classes": f1_score(labels, prediction, labels=present_labels, average="macro", zero_division=0),
        "creator_mean_macro_f1": float(np.mean(creator_scores)),
        "creator_min_macro_f1": float(np.min(creator_scores)),
        "unknown_recall": recall_score(labels, prediction, labels=["unknown"], average="macro", zero_division=0),
    }

raw_metrics = score_predictions(raw_prediction)
rows, predictions = [], {}
for exit_threshold, exercise_enter, exercise_hold, min_duration_s, merge_gap_s in product(
    [0.70, 0.80, 0.90], [0.25, 0.40], [0.15, 0.25], [0.0, 1.0, 2.0], [0.0, 0.6, 1.2]
):
    prediction = np.empty(len(labels), dtype=object)
    for group in sorted(CALIBRATION_GROUPS):
        indices = np.flatnonzero(groups == group)
        prediction[indices] = decode_creator(
            indices, exit_threshold, exercise_enter, exercise_hold, min_duration_s, merge_gap_s
        )
    metrics = score_predictions(prediction)
    key = (exit_threshold, exercise_enter, exercise_hold, min_duration_s, merge_gap_s)
    predictions[key] = prediction
    rows.append({
        "gate_enter": GATE_ENTER,
        "gate_exit": exit_threshold,
        "exercise_enter": exercise_enter,
        "exercise_hold": exercise_hold,
        "min_duration_s": min_duration_s,
        "merge_gap_s": merge_gap_s,
        **metrics,
    })

grid = pd.DataFrame(rows).sort_values(
    ["creator_mean_macro_f1", "creator_min_macro_f1", "macro_f1_present_classes"], ascending=False
).reset_index(drop=True)
best = grid.iloc[0]
best_key = tuple(float(best[name]) for name in [
    "gate_exit", "exercise_enter", "exercise_hold", "min_duration_s", "merge_gap_s"
])
selected_prediction = predictions[best_key]
comparison = pd.DataFrame([
    {"decoder": "raw_dense", **raw_metrics},
    {"decoder": "selected_dense", **score_predictions(selected_prediction)},
])
print("Selected decoder parameters:")
display(best[["gate_enter", "gate_exit", "exercise_enter", "exercise_hold", "min_duration_s", "merge_gap_s"]].to_frame().T)
print("Raw versus decoded:")
display(comparison)


## Export the decoder report

The archive keeps the entire grid, every dense prediction, the selected
parameters, confusion matrix, and creator timeline plots. No model weights are
changed.


In [ ]:
grid.to_csv(REPORT_ROOT / "decoder_grid.csv", index=False)
comparison.to_csv(REPORT_ROOT / "raw_vs_decoded.csv", index=False)

prediction_frame = pd.DataFrame({
    "group_id": groups,
    "clip_id": clip_ids,
    "center_s": centers_s,
    "true_class": labels,
    "gate_supported_probability": gate_probability,
    "exercise_top": exercise_top,
    "exercise_confidence": exercise_confidence,
    "raw_prediction": raw_prediction,
    "decoded_prediction": selected_prediction,
})
prediction_frame.to_csv(REPORT_ROOT / "dense_window_predictions.csv", index=False)

report = classification_report(
    labels, selected_prediction, labels=present_labels, output_dict=True, zero_division=0
)
(REPORT_ROOT / "decoded_classification_report.json").write_text(json.dumps(report, indent=2))
summary = {
    "experiment": "v6_dense_decoder_calibration",
    "provisional": True,
    "evidence_note": "Historical annotation notes retain a draft warning pending visual spot-check.",
    "model": "frozen_v6c",
    "dense_stride_frames": DENSE_STRIDE,
    "dense_step_seconds": STEP_SECONDS,
    "calibration_groups": sorted(CALIBRATION_GROUPS),
    "calibration_segments": int(len(manifest)),
    "calibration_windows": int(len(labels)),
    "present_classes": present_labels,
    "selected_parameters": {
        name: float(best[name]) for name in [
            "gate_enter", "gate_exit", "exercise_enter", "exercise_hold", "min_duration_s", "merge_gap_s"
        ]
    },
    "raw_metrics": {key: float(value) for key, value in raw_metrics.items()},
    "decoded_metrics": {key: float(value) for key, value in score_predictions(selected_prediction).items()},
    "locked_test_used": False,
}
(REPORT_ROOT / "decoder_summary.json").write_text(json.dumps(summary, indent=2))

matrix = confusion_matrix(labels, selected_prediction, labels=present_labels)
plt.figure(figsize=(9, 8))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=present_labels, yticklabels=present_labels)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Provisional dense decoder confusion matrix")
plt.tight_layout()
plt.savefig(REPORT_ROOT / "decoded_confusion_matrix.png", dpi=180)
plt.show()

timeline_labels = ["unknown"] + [label for label in SUPPORTED_CLASSES if label != "unknown"]
label_order = {label: index for index, label in enumerate(timeline_labels)}
fig, axes = plt.subplots(len(CALIBRATION_GROUPS), 1, figsize=(15, 8), sharey=True)
for axis, group in zip(axes, sorted(CALIBRATION_GROUPS)):
    mask = groups == group
    axis.step(centers_s[mask], [label_order[x] for x in labels[mask]], where="mid", label="truth", linewidth=2)
    axis.step(centers_s[mask], [label_order[x] for x in selected_prediction[mask]], where="mid", label="decoded", alpha=0.8)
    axis.set_title(group)
    axis.grid(alpha=0.2)
axes[-1].set_yticks(range(len(timeline_labels)), timeline_labels)
axes[-1].set_xlabel("Video time (seconds)")
axes[0].legend(loc="upper right")
plt.tight_layout()
plt.savefig(REPORT_ROOT / "creator_timelines.png", dpi=180)
plt.show()

zip_path = Path(shutil.make_archive(str(REPORT_ROOT), "zip", root_dir=REPORT_ROOT))
print("Decoder report:", zip_path.resolve())
print("ZIP size MB:", round(zip_path.stat().st_size / 1024**2, 2))
if IN_COLAB:
    files.download(str(zip_path))


## What to return

Send back `calorie_v6_dense_decoder_report.zip`. We will check whether temporal
stabilisation improves creator-equal Macro F1 without destroying short valid
exercises or unknown recall. We will not run the locked long-video test until
the decoder settings are frozen.
